In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# プリミティブの概要

<Accordion>
<AccordionItem title="Package versions">

このページのコードは以下の要件を使用して開発されました。
これらのバージョン以降を使用することをお勧めします。

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## Qiskitがプリミティブを導入した理由
古典コンピューターの黎明期、開発者がCPUレジスタを直接操作しなければならなかったように、QPUへの初期インターフェースは制御エレクトロニクスからの生データをそのまま返すだけのものでした。
QPUが研究室にあり、研究者だけが直接アクセスできた頃は、これは大きな問題ではありませんでした。
しかし、多くの開発者がそのような生データを0と1に変換する方法を知っているべきではないと認識し、Qiskitはクラウド上のQPUにアクセスするための最初の抽象化として`backend.run`を導入しました。これにより、開発者は使い慣れたデータ形式で操作し、より大きな目標に集中できるようになりました。

QPUへのアクセスがより広まり、より多くの量子アルゴリズムが開発されるにつれて、
より高レベルな抽象化の必要性が再び生じました。これに応えるため、Qiskitは
プリミティブインターフェースを導入しました。これは量子アルゴリズム開発における2つのコアタスクに最適化されています：
期待値推定（`Estimator`）と回路サンプリング（`Sampler`）です。目標は、
開発者がデータ変換よりもイノベーションに集中できるようにすることです。プリミティブインターフェースは`backend.run`インターフェースに取って代わります。`Sampler`が`backend.run`で提供されていたのと同じ直接的なハードウェアアクセスを提供するためです。

## プリミティブとは何か？
コンピューティングシステムは複数の抽象化レイヤーの上に構築されています。抽象化により、手元のタスクに関連する特定の詳細レベルに集中できます。ハードウェアに近づくほど、必要な抽象化のレベルは低くなります（例えば、CPUの命令レベルでデータを移動または操作する必要があるかもしれません）。実行したいタスクが複雑になるほど、抽象化のレベルは高くなります（例えば、代数計算を実行するためにプログラミングライブラリを使用するかもしれません）。

この文脈において、*プリミティブ*とは最小の処理命令であり、与えられた抽象化レベルで有用なものを作成できる最も単純な構成要素です。

量子コンピューティングの最近の進歩により、より高い抽象化レベルで作業する必要性が高まっています。
この分野が大規模な量子処理ユニット（QPU）やより複雑なワークフローに向かうにつれて、焦点は個々の量子ビット信号との対話から、必要なタスクを実行するシステムとして量子デバイスを見ることへとシフトします。

量子コンピューターの最も一般的な2つのタスクは、量子状態のサンプリングと期待値の計算です。
これらのタスクがQiskitプリミティブ、**Estimator**と**Sampler**の設計を動機づけました。

- Estimatorは量子回路によって準備された状態に関するオブザーバブルの期待値を計算します。
- Samplerは量子回路の実行から出力レジスタをサンプリングします。

端的に言えば、Qiskitプリミティブが導入した計算モデルは、量子プログラミングを今日の古典プログラミングに一歩近づけるものです。ハードウェアの詳細よりも、達成しようとしている結果に焦点を当てることができます。

## プリミティブの定義と実装
Qiskitプリミティブには2種類あります：基底クラスとその実装です。EstimatorおよびSamplerプリミティブは、Qiskit SDK（[`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)モジュール内）に含まれるオープンソースのプリミティブ基底クラスによって定義されています。プロバイダー（Qiskit Runtimeなど）はこれらの基底クラスを使用して、独自のSamplerおよびEstimatorの実装を派生させることができます。ほとんどのユーザーは基底プリミティブではなく、プロバイダーの実装を使用します。

### 基底クラス
`Base`プリミティブは、プリミティブを実装するための共通インターフェースを定義する抽象クラスです。[`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)モジュール内の他のすべてのクラスはこれらの基底クラスを継承します。特定のプロバイダー向けに独自のプリミティブベースの実行モデルを作成することに興味がある開発者はこれらを使用してください。これらのクラスは、高度にカスタマイズされた処理を行いたいが、既存のプリミティブ実装が自分のニーズに対して単純すぎると感じる方にも役立つかもしれません。一般ユーザーは基底クラスを直接使用しません。

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1)および[`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) — V1プリミティブは引き続き使用可能ですが、このガイドではV2プリミティブに焦点を当てています。V2プリミティブは最新版であり、より広く使用されているためです。

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2)および[`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) — Qiskitリファレンスプリミティブはこれらのインターフェース仕様に従います。

<span id="implementations"></span>
### 実装
すべてのプリミティブは基底クラスから作成されるため、同じ一般的な構造と使用方法を持ちます。例えば、すべてのEstimatorプリミティブへの入力フォーマットは同じです。ただし、それぞれを独自にする実装の違いがあります。

以下はプリミティブ基底クラスの実装です：

- [Qiskit Runtimeプリミティブ](/guides/qiskit-runtime-primitives)（[`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2)および[`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)）は、クラウドベースのサービスとして、より高度な実装（例えばエラー緩和を含む）を提供します。この基底プリミティブの実装は、IBM Quantum&reg;ハードウェアへのアクセスに使用されます。

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator)および[`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) — Qiskitに組み込まれたシミュレーターを使用するプリミティブのリファレンス実装です。Qiskitの[`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information)モジュールで構築されており、理想的な状態ベクトルシミュレーションに基づく結果を生成します。Qiskitを通じてアクセスします。使用方法の詳細は[Qiskitプリミティブによる厳密シミュレーション](/guides/simulate-with-qiskit-sdk-primitives)を参照してください。

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2)および[`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) — これらのクラスを使用して、任意の量子コンピューティングリソースをプリミティブに「ラップ」できます。これにより、まだプリミティブベースのインターフェースを持っていないプロバイダーに対してプリミティブスタイルのコードを記述できます。これらのクラスは通常のSamplerやEstimatorと同様に使用できますが、実行する量子コンピューターを選択するための追加の`backend`引数で初期化する必要があります。Qiskitを使用してアクセスします。詳細は[バックエンドプリミティブ](/guides/get-started-with-backend-primitives)ガイドを参照してください。

## オプション
プリミティブにオプションを渡して、ニーズに合わせてカスタマイズできます。プリミティブの`run()`メソッドのインターフェースはすべての実装で共通ですが、オプションは共通ではありません。特定のプリミティブ実装がサポートするオプションについては、各APIリファレンスを参照してください。

例えば、Qiskit Runtimeプリミティブのオプションについては[Estimatorオプション](/guides/estimator-options)および[Samplerオプション](/guides/sampler-options)トピックを参照し、Qiskit AerプリミティブのオプションについてはQiskit Aer APIリファレンス（[Qiskit Aer API references](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html)）を参照してください。

## Qiskitプリミティブの利点
プリミティブを使用することで、Qiskitユーザーはすべての詳細を明示的に管理することなく、特定のQPU向けに量子コードを記述できます。また、追加の抽象化レイヤーにより、特定のプロバイダーの高度なハードウェア機能にアクセスしやすくなる場合があります。例えば、Qiskit Runtimeプリミティブを使用すると、エラー緩和や抑制の最新の進歩を、プリミティブの[`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)などのオプションを切り替えるだけで活用できます。これらのテクニックを独自に実装する必要はありません。

ハードウェアプロバイダーにとって、プリミティブをネイティブに実装することは、高度なポストプロセッシングテクニックなどのハードウェア機能に「すぐに使える」形でアクセスできる方法をユーザーに提供できることを意味します。これにより、ユーザーがハードウェアの最高の機能から恩恵を受けやすくなります。

## 次のステップ
> **Tip:** - [プリミティブの入出力](/guides/primitive-input-output)を理解してください。
> - 詳細な[使用例](/guides/simulate-with-qiskit-sdk-primitives)を確認してください。
> - IBM Quantum Learningの[コスト関数のレッスン](/learning/courses/variational-algorithm-design/cost-functions)を通じてプリミティブを練習してください。
> - [プロバイダーの作成](/guides/create-a-provider)を確認して、独自のSamplerおよびEstimatorプリミティブを実装する方法を学んでください。
> - [APIリファレンス](https://docs.quantum.ibm.com/api/qiskit/primitives)を参照してください。
> - [V2プリミティブへの移行](/guides/v2-primitives)を読んでください。
> - IBM QPUでの回路実行に使用する[Qiskit Runtimeプリミティブ](/guides/qiskit-runtime-primitives)について学んでください。